# 컴파일러 진단: `warnings{...}`와 `remarks{...}`

컴파일러는 커널을 PTX로 번역하는 데서 그치지 않고, 커널을 *검사*해서 뭔가 이상해 보이면 알려줄 수도
있습니다. 이 검사는 옵트인(opt-in) 옵션 두 개로 켭니다:

- **`warnings{...}`**는 **합법적이지만 미심쩍은** 패턴을 보여줍니다. 컴파일되고 실행될 수도 있지만,
  멈추거나(hang) 폴트가 나거나 의도와 다르게 동작할 가능성이 실제로 있는 코드입니다.
- **`remarks{...}`**는 **성능 전용** 발견 사항을 보여줍니다. 프로그램은 기능적으로 맞지만 더 빠를 수
  있다는 뜻입니다(예: 레지스터가 로컬 메모리로 넘치는 spill).

둘 다 **치명적이지 않습니다**(non-fatal) — 컴파일은 그대로 성공하고 커널도 얻습니다. 세 번째 심각도인
**에러**는 *항상 켜져 있고* *항상 치명적*입니다. 옵션과 무관하게, 증명된 결함에서 발동합니다. 이번 장은
각 심각도의 예를 하나씩 처음부터 끝까지 보여주고, 검사를 켜는 정확한 방법을 알려줍니다.

원문: [`NVIDIA/cutlass` examples/python/CuTeDSL/notebooks/1_dsl_features/03_diagnostics.ipynb](https://github.com/NVIDIA/cutlass/blob/098de2a652cf8f00fd70b2df54051c7eccbb855a/examples/python/CuTeDSL/notebooks/1_dsl_features/03_diagnostics.ipynb) (BSD-3-Clause, commit `098de2a` 기준)

**배울 내용:** `cute.compile(..., options="warnings{nvvm}")` / `"remarks{ptx}"`(또는 같은 값을 주는
`CUTE_DSL_COMPILER_OPT` 환경변수)로 컴파일러 진단 검사를 켜는 법; 세 가지 진단 **심각도** —
*remark*(성능), *warning*(미심쩍음), *error*(결함) — 와 그중 어느 것이 치명적인지; 치명적 진단을
파이썬에서 `CompilerDiagnosticError`로 잡는 법.

**선수 지식:** 원문은 `cutlass.Array`와 메모리 공간을 다룬 챕터(`2_primitives`)를 선수로 적었습니다.
이 저장소는 아직 `2_primitives`를 번역하지 않았으니, 아래 커널에서 쓰이는 `cutlass.Array`,
`mbarrier` 같은 이름은 지금은 "그런 게 있다" 정도로만 읽으면 충분합니다 — 이번 장의 요점은
커널 내용이 아니라 *진단을 켜는 방법*입니다.

**실행 환경:** 이 노트북의 셀은 전부 *컴파일만* 합니다(가짜 배열 `make_fake_compact_array`를 쓰고 커널을
launch하지 않습니다). 원문은 섹션 1·2의 동기화 프리미티브(`elect.sync`, `cp.async.bulk`, `mbarrier`)가
**sm_90 이상**을 대상으로 한다고 적었고, 섹션 3(ptxas 리마크)은 어느 타깃에서든 동작한다고 적었습니다.
이 저장소의 이전 두 노트북은 Colab **L4**(sm_89)에서 검증했는데 L4는 sm_90 미만이라, 이 노트북은
아래 "0-2" 셀에서 컴파일 타깃을 `sm_90a`로 덮어씌워 실행했습니다(자세한 건 그 절 참고).
`Arch` enum이 `sm_80`부터 시작하므로 T4는 해당 없습니다(README 참고).

## 0. 환경 확인

재현성을 위해 먼저 GPU 정보를 남깁니다.

In [ ]:
!nvidia-smi

## 0-1. CuTe DSL 설치

`nvidia-cutlass-dsl==4.8.0`으로 고정합니다(노트북1과 동일 버전).

**함정(노트북1에서 이미 확인함): 같은 커널에서 설치 직후 import하면 실패합니다.** `.pth`
파일이 인터프리터 시작 시점에만 읽히기 때문 — 설치 셀 다음에 반드시 `site.main()` 셀을
넣습니다.

In [ ]:
!pip install -q nvidia-cutlass-dsl==4.8.0

In [ ]:
import site
site.main()  # 방금 설치된 nvidia-cutlass-dsl의 .pth 경로를 sys.path에 반영

## 0-2. 컴파일 타깃 아키텍처 지정 (`CUTE_DSL_ARCH`)

`cute.compile`은 기본적으로 **지금 붙어 있는 GPU의 아키텍처**를 컴파일 타깃으로 씁니다(L4면
`sm_89`). 그런데 섹션 1의 `elect.sync`/`cp.async.bulk`는 원문이 sm_90 이상 대상이라고 적은
명령어입니다. 이 노트북은 컴파일만 하고 실행은 하지 않으므로, **`import cutlass` 전에** 환경변수
`CUTE_DSL_ARCH`를 `sm_90a`(Hopper)로 지정해서 타깃을 덮어씁니다.

**실측(Colab L4, 새 커널):** 이 셀 없이 실행하면 `detect_gpu_arch`가 `sm_89`를 돌려주고, 섹션 1의
첫 `cute.compile`(옵션 없는 컴파일)이 `CompilerDiagnosticError: NVVM backend compilation failed ...
target architecture: sm_89 ... Error: unsupported operation`으로 실패합니다. 같은 조건에서 섹션 2는
원문의 `error[nvvm-diag:C3/C4]`가 그대로 나왔고 섹션 3(리마크)도 원문대로 동작했습니다. 다만 섹션 2가
`sm_89`에서 나온 건 진단 검사기가 백엔드 컴파일보다 먼저 에러를 내기 때문일 가능성이 큽니다(추정) —
섹션 2의 커널도 섹션 1과 같은 종류의 sm_90+ 명령어를 쓰므로, "직접 해보기"의 2번(옵션 빼기)·3번(버그
고친 뒤 재컴파일)처럼 검사기를 통과하는 변형은 `sm_89` 그대로는 섹션 1처럼 실패할 수 있고, 이 변형들은
`sm_89`에서 실행해보지 않았습니다. 그래서 이 노트북은 전체에 오버라이드를 유지합니다.

> 이 값은 `import cutlass` *이전*에 설정해야 합니다 — 환경변수가 임포트 시점에 읽힙니다(소스 확인).
> 컴파일 타깃만 바뀌고 GPU가 바뀌는 건 아니므로, 여기서 만든 결과물을 이 GPU에서 *실행*하지는
> 않습니다(이 노트북은 컴파일만 합니다).

실제 sm_90 GPU(H100 등)에서 돌린다면 이 셀은 필요 없습니다(감지된 `sm_90a`를 그대로 씁니다).
다만 그 경우는 직접 검증하지 못했습니다 — 아래 실측은 **L4 + 타깃 오버라이드** 조합입니다.

In [ ]:
import os

# import cutlass 전에 지정해야 합니다. 실제 sm_90+ GPU라면 이 줄은 생략해도 됩니다.
os.environ["CUTE_DSL_ARCH"] = "sm_90a"
print("CUTE_DSL_ARCH =", os.environ["CUTE_DSL_ARCH"])

In [ ]:
import cutlass
import cutlass.cute as cute
from cutlass.experimental import primitives as prims 
from cutlass.base_dsl.compiler import CompilerDiagnosticError

## 레벨과 카테고리

`cute.compile`에 **옵션 문자열**을 넘겨서 진단을 켭니다. 문자열에는 두 개의 축이 있습니다: 보여줄
심각도 레벨과, 수집할 진단 카테고리입니다. 경고(warning)와 리마크(remark)는 옵트인이고 치명적이지
않습니다. 켜진 카테고리가 보고한 에러는 항상 표시되고 컴파일을 실패시킵니다 — 별도의
`errors{...}` 옵션은 없습니다.

| 레벨 | 켜는 법 | 용도 | 치명적? |
|---|---|---|---|
| Info (remark) | `"remarks"` 또는 `"remarks{<category>}"` | 성능 전용 발견 사항. 동기화 기회, ptxas 리소스 보고 포함. | 아니오 |
| Warning | `"warnings"` 또는 `"warnings{<category>}"` | 합법적이지만 미심쩍은 패턴. 멈추거나, 폴트가 나거나, 의도와 다르게 동작할 수 있음. | 아니오 |
| Error | 해당 카테고리를 `"warnings{<category>}"` 또는 `"remarks{<category>}"`로 켬. | 켜진 진단 카테고리가 보고한, 증명된 결함. | 예 |

| 카테고리 | 켜는 법 | 출처 | 유용한 레벨 |
|---|---|---|---|
| `nvvm` | `"warnings{nvvm}"`, `"remarks{nvvm}"` | `mbarrier`, bulk copy, TMA multicast, `tcgen05` 같은 연산에 대한 NVVM 수준 프리미티브 프로토콜 진단. | Error, warning, info (remark) |
| `ptxas` (선택자: `ptx`) | `"remarks{ptx}"` | remark 스트림으로 올라오는 ptxas 리소스 진단. 레지스터 spill, 로컬 메모리 사용량 포함. | Info (remark) |

똑같은 문자열이 환경변수로도 통합니다. 코드를 고치지 않고 진단을 켜고 싶을 때 편합니다:

```bash
CUTE_DSL_COMPILER_OPT="warnings{nvvm}" python my_kernel.py
```

메커니즘은 이게 전부입니다. 이번 장은 구체적인 발견 사항이 아니라 *검사를 켜는 방법*을 다루므로,
아래 세 예제는 각 심각도를 하나씩 처음부터 끝까지 보여줍니다.

## 1. 경고(warning) — 합법적이지만 미심쩍음

*경고*는 컴파일러가 기꺼이 빌드해주지만 실행 시점에 실제 위험이 있는 코드에서 발동합니다. 여기서
커널은 균일한(uniform) bulk copy를 full-mask `elect.sync`로 가드합니다 — 올바른 단일 발행자
관용구입니다. 그런데 실행(launch)은 스레드 4개짜리 **부분 워프**(partial warp)를 씁니다. 뒤쪽의
부분 워프에서 full-mask elect는 멈추거나 폴트가 날 수 있어서 컴파일러가 표시합니다. *틀렸다고
증명된* 것이 아니라 *미심쩍은* 것이므로 **경고**입니다: 컴파일은 그대로 성공합니다.

In [ ]:
@cute.kernel
def partial_warp_bulk_copy_kernel(gmem: cutlass.Array):
    # A single-issuer bulk copy: one elected lane drives the cp.async.bulk for the whole warp.
    mbar = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem, alignment=8)
    smem_dst = cutlass.Array(cutlass.Int32, 4, space=cutlass.AddressSpace.smem, alignment=16)

    if prims.elect_sync():  # full-mask elect: one lane of the (assumed full) warp
        prims.cp_async_bulk_shared_cluster_global(smem_dst, gmem, mbar, 16)


@cute.jit
def warning_host(gmem: cutlass.Array):
    # block=(4,1,1) is only a PARTIAL warp -> the elect.sync above becomes questionable.
    partial_warp_bulk_copy_kernel(gmem).launch(grid=(1, 1, 1), block=(4, 1, 1))


# A fake (compile-only) array + stream let us compile without launching on a real GPU.
gmem = cutlass.runtime.make_fake_compact_array(cutlass.Int32, (4,), assumed_align=16)

print(">>> compiling WITHOUT warnings{nvvm} (no diagnostic):")
cute.compile(warning_host, gmem)
print("    compiled, silent.\n")

print(">>> compiling WITH options='warnings{nvvm}':")
cute.compile(warning_host, gmem, options="warnings{nvvm}")
print("    compiled anyway -- the warning is non-fatal.")

# Expected output: the first compile is silent; the second prints a source-located
# warning before succeeding (compilation does NOT fail):
#
# warning[nvvm-diag:C13]: full-mask elect.sync guards ... in a partial-warp launch block
#   --> .../03_diagnostics...:NN:7
#      ...
#   suggestion: launch with a warp-multiple block size ...

## 2. 에러(error) — 증명된 결함, 항상 치명적

*에러*는 컴파일러가 증명할 수 있는 결함에서 발동하고, **언제나** 컴파일을 실패시킵니다. "errors"
플래그는 없습니다: 컴파일러의 검사기(checker)는 `warnings{nvvm}`(또는 `remarks{nvvm}`)을 요청하면
돌고, 일단 돌기만 하면 발견한 에러는 *어떤 심각도를 보겠다고 했든* 치명적입니다.

이 커널은 `mbarrier`를 위상(phase)당 도착(arrival) **정확히 한 번**을 기대하도록 초기화한 뒤,
*모든* 스레드가 — 가드 없이 — 도착시킵니다. 배리어가 과다 도착(over-arrive)해서 위상이 예기치 않게
뒤집히는, 스타일 문제가 아니라 진짜 버그입니다. 치명적이므로 컴파일을 `try/except`로 감싸
`CompilerDiagnosticError`를 잡습니다.

In [ ]:
@cute.kernel
def unguarded_arrive_kernel():
    mbar = cutlass.Array(cutlass.Int64, 1, space=cutlass.AddressSpace.smem, alignment=8)

    if prims.elect_sync():
        prims.mbarrier_init(mbar, 1)  # expect exactly ONE arrival per phase
    prims.fence_mbarrier_init()
    cute.arch.barrier()

    # BUG: no single-issuer guard, so all 32 threads arrive on a count=1 barrier.
    prims.mbarrier_arrive(mbar, count=1)
    prims.mbarrier_try_wait_parity(mbar, 0, time_limit=10_000_000)


@cute.jit
def error_host():
    unguarded_arrive_kernel().launch(grid=(1, 1, 1), block=(32, 1, 1))


print(">>> compiling WITH options='warnings{nvvm}' (runs the checkers):")
try:
    # warnings{nvvm} runs the checkers; any error they find is fatal no matter which severity you ask for.
    cute.compile(error_host, options="warnings{nvvm}")
    print("    unexpected: compile succeeded")
except CompilerDiagnosticError as exc:
    print("    compilation FAILED, as it should -- here is the diagnostic:\n")
    print(exc)

## 3. 리마크(remark) — 맞지만 더 느림

*리마크*는 성능 전용입니다: 프로그램은 기능적으로 맞지만, 컴파일러가 성능을 깎는 무언가를
발견했습니다. 대표 예가 **레지스터 spill**입니다 — 커널이 한꺼번에 너무 많은 값을 살려두면
`ptxas`가 레지스터가 모자라 넘치는 값을 로컬 메모리에 치우고, 그만큼 트래픽과 지연이 늘어납니다.

이 커널은 64개짜리 윈도우를 레지스터에 로드한 뒤 **뒤집어서** 다시 쓰는데, 그러면 64개 값이
동시에 전부 살아 있어야 합니다. spill을 강제로 일으키려고 일부러 빡빡한 레지스터 예산
(`--maxrregcount=24`)으로 컴파일하고, `ptx` 리마크 카테고리를 요청합니다. `--remark-output`은
설정하지 않고 두세요 — 그래야 파이썬 진단 렌더러가 소스 프레임을 출력합니다. `ptxas`는 spill 합계를
커널 단위로 보고하기 때문에, 소스 프레임은 정확한 spill 명령어가 아니라 보고된 커널 / 레지스터 압력이
높았을 법한 영역을 가리킵니다.

In [ ]:
_N = 64  # values per thread -- a full reversed window kept live forces register pressure


@cute.kernel
def reverse_store_kernel(src: cutlass.Array, dst: cutlass.Array):
    tid, _, _ = cute.arch.thread_idx()
    base = tid * cutlass.Int32(_N)
    window = src[base:_N]  # load the whole V-wide window into registers
    # Storing it reversed keeps every element of `window` live at once -> high register pressure.
    for i in cutlass.range_constexpr(_N):
        dst[base + cutlass.Int32(i)] = window[_N - 1 - i]


@cute.jit
def remark_host(src: cutlass.Array, dst: cutlass.Array):
    reverse_store_kernel(src, dst).launch(grid=(1, 1, 1), block=(32, 1, 1))


n = _N * 32
src = cutlass.runtime.make_fake_compact_array(cutlass.Int32, (n,), assumed_align=16)
dst = cutlass.runtime.make_fake_compact_array(cutlass.Int32, (n,), assumed_align=16)

print(">>> compiling WITH the ptx remark category + a tight register budget:")
# remarks{ptx} asks ptxas for perf remarks; --ptxas-options squeezes the register budget
# so the spill actually happens.
cute.compile(
    remark_host,
    src,
    dst,
    options=(
        "remarks{ptx} "
        "--ptxas-options '--maxrregcount=24 --override-directive-values'"
    ),
)
print("    compiled (the spill is perf-only, never fatal). See remark[ptxas] above.")

# Expected output: compilation succeeds and prints a ptxas spill remark with a source frame.

### 실측 검증 (2026-10-01, Colab L4)

이 노트북 전체를 `.ipynb` 단위로(실제 셀 순서대로 하나의 커널에서) Colab **L4**에서 실행해
확인했습니다. 코드 셀 8개가 전부 에러 없이 통과했고(`nvidia-cutlass-dsl==4.8.0`), 세 가지 진단이 모두
실제로 출력됐습니다:

| 섹션 | 확인한 출력 | 원문 기대 출력과 |
|---|---|---|
| 1 경고 | 옵션 없는 컴파일은 조용, `warnings{nvvm}` 컴파일은 `warning[nvvm-diag:C13]: full-mask elect.sync guards ... in a partial-warp launch block` 출력 후 성공 | 일치 |
| 2 에러 | `CompilerDiagnosticError`로 잡힘. 본문은 `error[nvvm-diag:C3/C4]: mbarrier arrive reaches a count=1 barrier from multiple threads` (ptxas 실패가 아니라 진짜 over-arrive 진단임을 본문으로 확인) | 일치 |
| 3 리마크 | `remark[ptxas]: ptxas detected register spills` — `288 bytes stack frame, 280 bytes spill stores, 280 bytes spill loads`, `Used 24 registers` | 일치 |

**출력 순서 주의(관찰):** 경고와 리마크는 `stderr`로, 셀의 `print`는 `stdout`으로 나옵니다.
`colab exec`로 저장한 출력에서는 섹션 1의 경고 블록이 `>>> compiling ...` 줄들보다 *위에* 찍혔습니다
(stderr 스트림이 먼저, 그 뒤에 stdout 전체). 원문 주석처럼 "두 번째 컴파일 직후"에 나오는 게 아니라,
셀 출력 맨 위에 모여 보일 수 있습니다. 웹 UI에서의 순서는 확인하지 않았습니다.

**이 검증의 한계:** 컴파일 타깃을 `sm_90a`로 덮어쓴 L4 검증입니다. 실제 sm_90 GPU에서 돌려본 게
아니고, 커널을 실행하지도 않았습니다(원문도 컴파일만 합니다).

## 직접 해보기

1. **끄기 vs. 드러내기.** 섹션 1을 `"warnings{nvvm}"` 대신 `options="remarks{nvvm}"`로 다시
   실행해 보세요: 부분 워프 *경고*가 더는 출력되지 않습니다(`warnings{nvvm}`에만 걸려 있습니다). 그래도
   컴파일은 성공합니다. 심각도 옵션은 컴파일러가 무엇을 *검사하는지*가 아니라 사용자가 무엇을
   *보는지*를 조절합니다.
2. **에러는 무조건입니다.** 섹션 2에서 `options=` 인자를 통째로 빼보세요. 과다 도착은 *결함*이니
   실패해야 할 것 같지만... 사정이 다릅니다: 진단 옵션이 없으면 동기화 검사기가 아예 돌지 않아서
   컴파일이 *조용히 성공*합니다. `options="remarks{nvvm}"`을 더하면 같은 결함이 이번엔 컴파일을
   실패시킵니다 — 진단을 요청하는 것(어느 심각도든)이 검사기를 돌리는 것이고, 돌기만 하면 에러는
   항상 치명적이라는 증거입니다.
3. **버그 고치기.** 섹션 2의 arrive를 `if prims.elect_sync():`(또는 `if tid == 0:`, `tid`는
   `cute.arch.thread_idx()`에서 읽음)로 감싸서 스레드 하나만 도착하게 하세요.
   `options="warnings{nvvm}"`로 다시 컴파일하면 에러가 사라집니다.
4. **spill을 악화시키기(또는 없애기).** 섹션 3에서 `_N`을 96으로 올리면 보고서의 spill 수치가 커지는
   걸 볼 수 있습니다. 반대로 `--ptxas-options ...` 부분을 빼서 ptxas가 레지스터 예산을 다 쓰게 하면
   spill 리마크가 사라집니다 — spill할 게 없었으니까요.
5. **환경변수 쓰기.** `options="..."` 대신 셸에서 `CUTE_DSL_COMPILER_OPT="warnings{nvvm}"`을
   설정하고 옵션 문자열 없이 평범한 `cute.compile(...)`을 실행해 보세요 — 코드 변경 없이 같은
   진단이 나옵니다.